# Swarm Clock, Part 2a: wiki introductions, families, eligibility
Run in the same Colab runtime as Part 1 (reuses data/cw_revisions.jsonl.gz).
Frozen choices (do not change after looking at temporal results):
- Unit: introduction = text added by a revision (insert/replace hunks), or full body if the page was created.
- Window: time < 2026-07-03 UTC (before post-publication fakes), time_grade == reqlog.
- Masking: digits, long tokens, URLs' paths, agent labels, dates.
- Families: MinHash LSH (k=5 token shingles, threshold 0.5) on masked introductions, kept only if they span >= 2 pages.
- Eligibility: >= 15 dependency blocks (family x page), one date-blind representative per block (the longest introduction).

In [ ]:
!pip -q install datasketch
import gzip, json, re, collections, datetime as dt, base64
import pandas as pd, numpy as np
from datasketch import MinHash, MinHashLSH
def read_jsonl_gz(p):
    with gzip.open(p, "rt", encoding="utf-8", errors="replace") as f:
        for line in f:
            line=line.strip()
            if line:
                try: yield json.loads(line)
                except json.JSONDecodeError: pass
cw = list(read_jsonl_gz("data/cw_revisions.jsonl.gz"))
print(len(cw))
print("body_encoding:", collections.Counter(str(r.get("body_encoding")) for r in cw).most_common(5))
print("hunk ops:", collections.Counter(h.get("op") for r in cw for h in (r.get("hunks") or [])).most_common())
print("diff_base examples:", [ (r["rev_id"], r.get("diff_base"), r.get("diff_base_reason")) for r in cw if r.get("diff_base") is not None][:5])

In [ ]:
def get_body(r):
    b = r.get("body") or ""
    enc = str(r.get("body_encoding") or "").lower()
    if "base64" in enc:
        try: b = base64.b64decode(b).decode("utf-8", "replace")
        except Exception: pass
    return b

by_rev = {r["rev_id"]: r for r in cw}
by_page_seq = {(r["page_key"], r["seq"]): r for r in cw}
def resolve_base(r):
    db = r.get("diff_base")
    if db is None: return None
    if isinstance(db, str) and db in by_rev: return by_rev[db]
    if isinstance(db, (int, float)) or (isinstance(db, str) and db.isdigit()):
        return by_page_seq.get((r["page_key"], int(db)))
    if isinstance(db, dict):
        for k in ("rev_id", "seq"):
            if k in db: return resolve_base({**r, "diff_base": db[k]})
    return None

def introduction(r):
    body = get_body(r)
    hunks = r.get("hunks") or []
    if r.get("diff_base_reason") == "page_created" or not hunks:
        return body, "created" if r.get("diff_base_reason") == "page_created" else "no_hunks"
    lines = body.split("\n")
    added, bad = [], 0
    for h in hunks:
        if h.get("op") in ("insert", "replace"):
            b0, b1 = int(h.get("b0", 0)), int(h.get("b1", 0))
            if b1 > len(lines): bad += 1
            added.extend(lines[b0:b1])
    return "\n".join(added), ("bad_index" if bad else "hunks")

intro_rows = []
for r in cw:
    text, how = introduction(r)
    intro_rows.append({"rev_id": r["rev_id"], "page_id": r["page_id"], "label": r.get("label") or "",
                       "time": r.get("time"), "time_grade": r.get("time_grade"), "how": how,
                       "intro": text, "body_len": len(get_body(r))})
I = pd.DataFrame(intro_rows)
I["t"] = pd.to_datetime(I["time"], utc=True, errors="coerce")
print(I.how.value_counts())
print("unparseable times:", I.t.isna().sum())
# Sanity: introductions must be much shorter than bodies for non-created revisions
nc = I[I.how=="hunks"]
print("median intro/body length ratio (hunks):", (nc.intro.str.len()/nc.body_len.clip(lower=1)).median())

In [ ]:
# Manual audit: 8 random non-created revisions. The introduction should be only the new text.
for _, x in nc.sample(8, random_state=1).iterrows():
    print("----", x.rev_id, x.t, "| label:", x.label, "| body_len", x.body_len)
    print(x.intro[:400])

In [ ]:
# Window and quality filters
CUTOFF = pd.Timestamp("2026-07-03", tz="UTC")
F = I[(I.t < CUTOFF) & (I.time_grade == "reqlog") & (I.how != "bad_index")].copy()
print("after filters:", len(F))
# Candidate front/hub pages: the most-edited pages. Inspect and fill EXCLUDE_PAGES if one is a main page.
print(F.page_id.value_counts().head(10))
EXCLUDE_PAGES = []   # e.g. ["dse/StartSeite"] after inspecting the list above
F = F[~F.page_id.isin(EXCLUDE_PAGES)]

In [ ]:
# Masking: agent labels, dates, digits, long tokens, URL paths
labels = sorted({l for l in F.label if isinstance(l, str) and len(l) >= 4}, key=len, reverse=True)
re_labels = re.compile(r"(?<![A-Za-z0-9])(" + "|".join(map(re.escape, labels)) + r")(?![A-Za-z0-9])") if labels else None
re_url  = re.compile(r"(https?://[^/\s'\"]+)[^\s'\"]*")
re_date = re.compile(r"(?i)\b(19|20)\d\d[-/.]\d{1,2}[-/.]\d{1,2}\b|\b(jan|feb|mar|apr|may|jun|jul|aug|sep|oct|nov|dec)[a-z]*\.?\s?\d{1,2}\b")
re_long = re.compile(r"[A-Za-z0-9+/=_-]{24,}")
re_num  = re.compile(r"\d+")
def mask(t):
    if re_labels is not None: t = re_labels.sub(" <AGENT> ", t)
    t = re_url.sub(r"\1/<PATH>", t)
    t = re_date.sub(" <DATE> ", t)
    t = re_long.sub(" <TOK> ", t)
    t = re_num.sub("0", t)
    return re.sub(r"\s+", " ", t).strip()
F["m"] = F.intro.map(mask)
F["ntok"] = F.m.str.split().str.len()
F = F[F.ntok >= 8].reset_index(drop=True)
print("introductions with >= 8 tokens:", len(F))

In [ ]:
K, NPERM, THR = 5, 128, 0.5
def shingles(t):
    toks = t.split()
    return {" ".join(toks[j:j+K]) for j in range(max(1, len(toks)-K+1))}
F["sh"] = F.m.map(shingles)
lsh = MinHashLSH(threshold=THR, num_perm=NPERM); mh = []
for i, s in enumerate(F.sh):
    m = MinHash(num_perm=NPERM)
    for x in s: m.update(x.encode())
    mh.append(m); lsh.insert(str(i), m)
parent = list(range(len(F)))
def find(x):
    while parent[x] != x:
        parent[x] = parent[parent[x]]; x = parent[x]
    return x
for i in range(len(F)):
    for j in lsh.query(mh[i]):
        a, b = find(i), find(int(j))
        if a != b: parent[a] = b
F["family"] = [find(i) for i in range(len(F))]

In [ ]:
# Dependency blocks and date-blind representatives
F["block"] = F.family.astype(str) + "|" + F.page_id
F["len"] = F.m.str.len()
reps = F.sort_values(["block", "len", "rev_id"], ascending=[True, False, True]).drop_duplicates("block")
fam = reps.groupby("family").agg(blocks=("block","nunique"), pages=("page_id","nunique"),
                                 labels=("label","nunique"),
                                 t_min=("t","min"), t_max=("t","max"))
fam["span_h"] = (fam.t_max - fam.t_min).dt.total_seconds()/3600
fam["days"] = reps.groupby("family").t.apply(lambda s: s.dt.date.nunique())
eligible = fam[(fam.blocks >= 15) & (fam.pages >= 2)].sort_values("blocks", ascending=False)
print("families total:", len(fam), "| eligible:", len(eligible), "| representatives in eligible:", eligible.blocks.sum())
print(eligible.head(25).to_string())

In [ ]:
# Peek at eligible families (masked text) so we can name them in the report
for f in eligible.index[:10]:
    ex = reps[reps.family == f].sort_values("t").m.iloc[0]
    print(f"family {f}: blocks={eligible.loc[f,'blocks']} span_h={eligible.loc[f,'span_h']:.1f} days={eligible.loc[f,'days']}")
    print("   ", ex[:250])
reps[reps.family.isin(eligible.index)][["rev_id","page_id","label","t","family","block","m"]].to_parquet("data/wiki_reps.parquet")
F[["rev_id","page_id","label","t","family","block","m"]].to_parquet("data/wiki_intros.parquet")
print("saved")